# 🎬 Trình Tải & Ghép Phim Ngắn + Trích Xuất & Dịch Phụ Đề - Google Colab CLI Edition

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kinyias/downloader/blob/main/colab.ipynb)

Công cụ CLI thuần túy chạy trực tiếp trên **Google Colab**:
- 🌐 **Giao diện Web UI Trực Quan & Link Công Khai**: Tự động tạo đường link HTTPS (Cloudflare Tunnel, Localtunnel, Pinggy) để mở toàn bộ giao diện Web trên trình duyệt máy tính hoặc điện thoại.
- ⚡ **Chế độ CLI & Web linh hoạt**: Có thể dùng giao diện Web trực quan hoặc dòng lệnh CLI thuần túy tùy thích.
- 🎙️ **CapCut ASR & Dịch Phụ Đề Tiếng Việt**: Tự động nhận diện giọng nói, dịch sang tiếng Việt bằng LLM, tự động kiểm tra re-translate sạch chữ tiếng Trung, lọc bỏ segment 1 từ đứng một mình (`á`, `a`...), và làm sạch dấu câu cuối câu.
- ☁️ **Tự động Upload storage.to**: Tự động tải video FULL, file phụ đề gốc `.srt` và file phụ đề tiếng Việt `_vi.srt` lên storage.to nhận link tải trực tiếp ngay lập tức.
- 📊 **Thanh tiến trình (Progress Bar)**: Xem trực tiếp tốc độ, phần trăm xử lý video và phụ đề.
- 🎯 **Tùy chọn tải linh hoạt**: Chọn tải toàn bộ hoặc khoảng tập mong muốn (ví dụ `1-20`, `21-40`, `1,3,5-10`).
- ⚡ **Tăng tốc phần cứng GPU NVENC**: Ghép video siêu tốc ~150-300 fps trên NVIDIA Tesla T4.
- 💾 **Lưu trữ Google Drive**: Toàn bộ video và phụ đề được lưu vào `/content/drive/MyDrive/ShortDrama_Downloads`.

### 📌 Bước 1: Kiểm tra GPU & Kết nối Google Drive
> *Khuyến khích: Vào menu `Runtime` -> `Change runtime type` -> Chọn **T4 GPU** để tăng tốc độ ghép video lên gấp 10-20 lần!*

In [ ]:
# 1. Kiểm tra GPU
!nvidia-smi

# 2. Kết nối Google Drive để lưu video vĩnh viễn
from google.colab import drive
drive.mount("/content/drive")

!mkdir -p /content/drive/MyDrive/ShortDrama_Downloads
print("✅ Thư mục Google Drive đã sẵn sàng: /content/drive/MyDrive/ShortDrama_Downloads")

### 📌 Bước 2: Cài đặt mã nguồn & Kích hoạt FFmpeg CUDA NVENC

In [ ]:
# Tải / Cập nhật repository
!if [ -d "/content/downloader" ]; then cd /content/downloader && git pull; else git clone https://github.com/kinyias/downloader.git /content/downloader; fi
%cd /content/downloader

# Cài đặt FFmpeg hỗ trợ NVIDIA CUDA / NVENC & thư viện Python
!if ! ffmpeg -encoders 2>/dev/null | grep -q nvenc; then \
    git clone -q https://github.com/rokibulislaam/colab-ffmpeg-cuda.git /tmp/colab-ffmpeg-cuda && \
    cp -r /tmp/colab-ffmpeg-cuda/bin/. /usr/bin/; \
fi
!if ! which cloudflared >/dev/null 2>&1; then \
    wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && \
    chmod +x /usr/local/bin/cloudflared; \
fi
!pip install -q -r requirements.txt
!pip install -q vieneu
print("✅ Môi trường, FFmpeg CUDA (NVIDIA Tesla T4) & Cloudflare Tunnel đã sẵn sàng!")


### 🌐 Bước 3: Khởi Động Giao Diện Web Trực Quan (Web UI với Link Công Khai)
> *Chạy ô bên dưới để khởi động toàn bộ **Giao Diện Web (Web UI)** đầy đủ tính năng: Tìm kiếm phim, Tải hàng loạt, Ghép video, Dịch phụ đề, Lồng tiếng VieNeu-TTS, xem video lồng tiếng và tự động upload lên storage.to.*
>
> 🚀 **Colab sẽ tự động tạo và in ra đường link HTTPS công khai** (qua Cloudflare Tunnel, Localtunnel hoặc Colab Port Proxy) kèm nút bấm để bạn mở ngay trên trình duyệt điện thoại hoặc máy tính!

In [ ]:
# =========================================================================
# 🌐 KHỞI ĐỘNG GIAO DIỆN WEB UI & TẠO LINK TRUY CẬP CÔNG KHAI
# =========================================================================
# Script tự động kết nối Cloudflare Tunnel, Localtunnel, Pinggy và Colab Proxy
# Chọn 1 trong các đường link HTTPS được in ra để mở giao diện Web trên trình duyệt!

!python colab_runner.py --tunnel auto --port 5000


### 📌 Bước 4: (Tùy chọn CLI) Giao diện Menu Tương Tác (Interactive Menu)
> *Chạy ô bên dưới để mở Menu chọn chức năng: Tìm kiếm phim, Nhập ID tải & ghép, Chọn khoảng tập...*

In [ ]:
# Mở menu tương tác trực tiếp trên Colab
!python cli.py

### ⚡ Tùy chọn 1: Tải phim, Tự động ghép & Dịch phụ đề tiếng Việt
> *Tự động tải các tập đã chọn, ghép thành 1 file MP4 duy nhất, trích xuất phụ đề CapCut ASR, dịch tiếng Việt, lọc bỏ chữ Trung sót lại, lọc bỏ segment 1 từ, xóa dấu câu cuối câu và upload lên storage.to.*

In [ ]:
SERIES_ID = "7369168922572164134"   # Nhập Series ID phim hoặc link
EPISODES = "1-20"                   # Khoảng tập cần tải (vd: "1-20", "21-40", hoặc để trống "" để tải hết)
PROMPT = "ai_tong_hop_thong_minh"   # Preset prompt: ai_tong_hop_thong_minh, review_phim_co_trang, review_phim_hien_dai...
MODEL = ""                          # Model dịch thuật (để trống dùng gemini-lite mặc định hoặc: deepseek-chat, gpt-4o-mini)
CUSTOM_ENDPOINT = ""                # Custom LLM API endpoint (nếu có, vd: http://localhost:20128/v1)
CUSTOM_APIKEY = ""                  # Custom API key (nếu có)
AUTO_TRANSLATE = True               # Bật tự động dịch phụ đề sang tiếng Việt
AUTO_DUBBING = True                 # Bật tự động lồng tiếng video với VieNeu-TTS
DUBBING_VOICE = "Ngọc Huyền"         # Giọng đọc: Minh Quân, Minh Đức, Phạm Tuyên, Trúc Ly, Mai Anh, Adam, Thái Sơn...

cmd_parts = ["python cli.py download", f'"{SERIES_ID}"', "--merge"]
if EPISODES.strip():
    cmd_parts.append(f'-e "{EPISODES.strip()}"')
if not AUTO_TRANSLATE:
    cmd_parts.append("--no-translate")
elif PROMPT.strip():
    cmd_parts.append(f'--prompt "{PROMPT.strip()}"')
if MODEL.strip():
    cmd_parts.append(f'--model "{MODEL.strip()}"')
if CUSTOM_ENDPOINT.strip():
    cmd_parts.append(f'--endpoint "{CUSTOM_ENDPOINT.strip()}"')
if CUSTOM_APIKEY.strip():
    cmd_parts.append(f'--apikey "{CUSTOM_APIKEY.strip()}"')
if not AUTO_DUBBING:
    cmd_parts.append("--no-dubbing")
elif DUBBING_VOICE.strip():
    cmd_parts.append(f'--voice "{DUBBING_VOICE.strip()}"')

cmd = " ".join(cmd_parts)
print(f"🚀 Đang thực thi lệnh:\n{cmd}\n")
!{cmd}


### 🔍 Tùy chọn 2: Tìm kiếm phim theo tên / từ khóa

In [ ]:
# Nhập tên phim hoặc từ khóa
KEYWORD = "Tổng tài"

!python cli.py search "$KEYWORD"

### 🎬 Tùy chọn 3: Ghép các file video có sẵn trong Google Drive & Dịch phụ đề

In [ ]:
# Đường dẫn thư mục chứa các tập video lẻ trên Google Drive
FOLDER_PATH = "/content/drive/MyDrive/ShortDrama_Downloads/Ten_Thu_Muc_Phim"
OUTPUT_NAME = "Phim_Hoan_Chinh.mp4"
PROMPT = "ai_tong_hop_thong_minh"
MODEL = ""
CUSTOM_ENDPOINT = ""
CUSTOM_APIKEY = ""
AUTO_TRANSLATE = True
AUTO_DUBBING = True
DUBBING_VOICE = "Ngọc Huyền"

cmd_parts = ["python cli.py merge", f'"{FOLDER_PATH}"']
if OUTPUT_NAME.strip():
    cmd_parts.append(f'--output-name "{OUTPUT_NAME.strip()}"')
if not AUTO_TRANSLATE:
    cmd_parts.append("--no-translate")
elif PROMPT.strip():
    cmd_parts.append(f'--prompt "{PROMPT.strip()}"')
if MODEL.strip():
    cmd_parts.append(f'--model "{MODEL.strip()}"')
if CUSTOM_ENDPOINT.strip():
    cmd_parts.append(f'--endpoint "{CUSTOM_ENDPOINT.strip()}"')
if CUSTOM_APIKEY.strip():
    cmd_parts.append(f'--apikey "{CUSTOM_APIKEY.strip()}"')
if not AUTO_DUBBING:
    cmd_parts.append("--no-dubbing")
elif DUBBING_VOICE.strip():
    cmd_parts.append(f'--voice "{DUBBING_VOICE.strip()}"')

cmd = " ".join(cmd_parts)
print(f"🚀 Đang thực thi lệnh:\n{cmd}\n")
!{cmd}


### 📖 Bảng Tra Cứu Preset Prompt Dịch Thuật (`translation/prompts.json`)

| Tên Preset (`--prompt`) | Thể loại & Bối cảnh dịch thuật |
|---|---|
| `ai_tong_hop_thong_minh` | **Mặc định** — Tự suy luận ngữ cảnh (Hiện đại, Cổ trang, Anime, Game...), không thêm dấu câu thừa |
| `ke_truyen` | Video kể chuyện, review, tóm tắt phim — giọng dẫn ngôi thứ ba kết hợp thoại nhân vật |
| `review_phim_co_trang` | Phim cổ trang, cung đấu — xưng hô Trẫm/Thần/Bản vương/Bản cung, nghiêm ngặt tôn ti |
| `review_phim_kiem_hiep` | Phim kiếm hiệp, tiên hiệp — giang hồ, môn phái, cấp bậc tu luyện (Luyện Khí, Trúc Cơ, Kim Đan...) |
| `review_phim_xuyen_khong` | Phim xuyên không — tương phản giữa lời nói hiện đại và người cổ đại |
| `review_phim_hien_dai` | Phim hiện đại, tổng tài, đời thường, hành động, sinh tồn, hài hước |
| `review_phim_kinh_di` | Phim kinh dị, giật gân, tâm linh — câu ngắn lạnh lùng, tạo cảm giác rợn gáy |
| `review_anime_nhat_ban` | Anime, donghua hoạt hình — giữ kính ngữ Senpai/Sensei, tên Romaji |
| `review_phim_han_quoc` | Phim Hàn Quốc — giữ đúng kính ngữ banmal, tôn trọng quan hệ nhân vật |
| `review_phim_au_my_chuan` | Phim Âu Mỹ kinh điển — giữ nguyên 100% tên tiếng Anh, cách xưng hô tự nhiên |

---

### 🎙️ Danh Sách Giọng Đọc Tiếng Việt VieNeu-TTS (`vieneu_tts.py`)

| Giọng Đọc (Voice ID) | Giới tính | Vùng miền / Phong cách |
| :--- | :--- | :--- |
| **Ngọc Huyền** *(Mặc định)* | Nữ | Miền Bắc - Giọng đọc nữ truyền cảm, ngọt ngào |
| **Minh Quân** | Nam | Miền Bắc - Giọng đọc chuẩn, tự nhiên |
| **Minh Đức** | Nam | Miền Bắc - Trầm ấm, truyền cảm |
| **Phạm Tuyên** | Nam | Miền Bắc - Tự nhiên, linh hoạt, diễn cảm |
| **Xuân Vĩnh** | Nam | Miền Bắc - Rõ ràng, dứt khoát |
| **Anh Khôi** | Nam | Miền Bắc - Trẻ trung, năng động |
| **Mạnh Dũng** | Nam | Miền Bắc - Hùng hồn, mạnh mẽ |
| **Trúc Ly** | Nữ | Miền Bắc - Nhẹ nhàng, êm dịu |
| **Mai Anh** | Nữ | Miền Bắc - Tươi sáng, hiện đại |
| **Quỳnh Anh** | Nữ | Miền Bắc - Thanh thoát, diễn cảm |
| **Quang Sơn** | Nam | Miền Trung - Truyền cảm, sâu lắng |
| **Ngọc Trân** | Nữ | Miền Trung - Ngọt ngào, đằm thắm |
| **Adam** | Nam | Miền Nam - Hiện đại, phong cách podcast |
| **Thái Sơn** | Nam | Miền Nam - Đĩnh đạc, nam tính |
| **Thùy Dung** | Nữ | Miền Nam - Dịu dàng, mượt mà |
| **Mỹ Duyên** | Nữ | Miền Nam - Trong trẻo, tự nhiên |
